In [15]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

- `pandas` — pra manipular os dados
- `StandardScaler` — pra normalizar as métricas
- `KMeans` — o modelo em si
- `matplotlib` — pra visualizar os resultados

<br>
<hr>

In [16]:
df = pd.read_excel('../data/players.xlsx')

ws = df[df['Segment Name'] == 'Whole Session']

print(ws.shape)
print(ws['Athlete ID'].nunique())

(708, 34)
26


- `read_excel` — carrega o arquivo
- `df[df['Segment Name'] == 'Whole Session']` — filtra só as linhas do jogo inteiro, descartando First Half, Second Half, etc.
- Os prints são pra confirmar que os dados estão certos — você deve ver algo como (X linhas, 34 colunas) e 26 atletas

<br>
<hr>

In [17]:
from datetime import datetime

metrics = [
    'Session Load',
    'Distance (m)',
    'High Intensity Running (m)',
    'Sprint Distance (m)',
    'Top Speed (kph)',
    'Accelerations',
    'Decelerations',
    'Metres per Minute (m)'
]

# Calcular peso por data
ws = ws.copy()
ws['Start Date'] = pd.to_datetime(ws['Start Date'])
today = pd.Timestamp.today()
ws['weight'] = 1 / ((today - ws['Start Date']).dt.days + 1)

# Média ponderada por atleta
def weighted_mean(group):
    return pd.Series({
        col: (group[col] * group['weight']).sum() / group['weight'].sum()
        for col in metrics
    })

player_df = ws.groupby('Athlete ID').apply(weighted_mean).dropna()

print(player_df.shape)
print(player_df.head())

(26, 8)
            Session Load  Distance (m)  High Intensity Running (m)  \
Athlete ID                                                           
75075071      763.989164   7320.834330                  282.237181   
436974456    1437.155071   9426.448678                  337.584793   
679795059     788.115880   6583.738777                  243.980583   
1010711410    684.973879   5373.383454                  189.534468   
1020172514    714.926281   6903.408165                  266.143423   

            Sprint Distance (m)  Top Speed (kph)  Accelerations  \
Athlete ID                                                        
75075071              57.170865        27.088658      55.805655   
436974456             70.391514        28.414166      53.309366   
679795059             57.108682        26.370592      48.868271   
1010711410            38.946971        23.422768      43.273222   
1020172514            60.289269        25.251447      50.862687   

            Decelerations  Metr

- `metrics` — selecionamos só as colunas relevantes pro perfil físico do atleta
- `groupby('Athlete ID')` — agrupa os dados por atleta
- `.mean()` — calcula a média de cada métrica ao longo de todos os jogos daquele atleta
- `.dropna()` — remove atletas que tenham alguma métrica faltando

<br>

> *Na parte de calcular a média, é bom que seja considerado o tempo do jogo. Jogos mais recentes devem ter mais pesos que jogos muito antigos :)*

<br>
<hr>

In [18]:
scaler = StandardScaler()
X = scaler.fit_transform(player_df)

print(X[:5])

[[-0.13795391  0.35785433  0.30552073 -0.20263294  0.52511472  0.71303513
   0.57872237  0.19186312]
 [ 2.91643275  2.56895633  1.78002067  1.11047534  1.4870961   0.32974957
   1.89435299  2.40916085]
 [-0.02848265 -0.41616844 -0.71366237 -0.20880911  0.0039814  -0.35214557
  -0.28832641 -0.45473651]
 [-0.49647359 -1.68716048 -2.16414577 -2.0126761  -2.13538963 -1.21122124
  -2.08600565 -1.80417659]
 [-0.36056918 -0.08048417 -0.12322848  0.10709485 -0.80823321 -0.04591869
  -0.38009311 -0.12924667]]


- `StandardScaler` transforma cada métrica pra ter média 0 e desvio padrão 1
- Isso é necessário porque as métricas têm escalas muito diferentes — Distance está na casa dos 7000, enquanto Top Speed está na casa dos 27. Sem normalizar, Distance dominaria o agrupamento injustamente
- `fit_transform` aprende a escala de cada coluna e já aplica a transformação
- O resultado `X` é um array numpy — as colunas não têm mais nome, mas a ordem é a mesma das métricas

<br>

>*Há um atleta com muitos dados negativos, depois tentar analisar caso ele tenha poucos jogos*

<br>
<hr>

In [19]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
km.fit(X)

labels = km.labels_

print(labels)

[1 3 2 0 2 2 1 2 2 3 1 1 1 1 1 2 2 1 2 2 2 1 1 1 1 0]


- `n_clusters=4` — pedimos 4 grupos, como definimos pelo cotovelo               *`src/calcular-K.ipynb`*
- `random_state=42` — garante que o resultado seja sempre o mesmo toda vez que rodar
- `n_init=10` — o algoritmo roda 10 vezes com centróides iniciais diferentes e escolhe o melhor resultado, evitando que caia num agrupamento ruim por azar
- `km.fit(X)` — aqui o modelo de fato aprende os clusters
- `labels` — é um array com o número do cluster de cada atleta (0, 1, 2 ou 3)

<br>
<hr>

In [20]:
player_df['Cluster'] = labels

print(player_df['Cluster'].value_counts())

Cluster
1    12
2    10
3     2
0     2
Name: count, dtype: int64


- Estamos adicionando uma coluna `Cluster` no dataframe com o número do grupo de cada atleta
- `value_counts()` mostra quantos atletas ficaram em cada cluster

In [21]:
print(player_df.groupby('Cluster')[metrics].mean().round(2))

         Session Load  Distance (m)  High Intensity Running (m)  \
Cluster                                                           
0              703.72       5493.71                      195.09   
1              822.21       7251.45                      284.56   
2              677.30       6451.66                      256.17   
3             1303.60       9479.96                      336.72   

         Sprint Distance (m)  Top Speed (kph)  Accelerations  Decelerations  \
Cluster                                                                       
0                      39.64            23.49          39.91          39.24   
1                      62.49            27.03          52.97          57.96   
2                      56.39            25.80          48.96          55.45   
3                      73.21            28.12          62.59          73.01   

         Metres per Minute (m)  
Cluster                         
0                        52.47  
1                      

Cluster 3 — "Elite" (2 atletas)
Dominante em tudo — Distance 9479m, Session Load 1303, Metres per Minute 87. São os atletas mais completos e intensos do elenco nas partidas recentes. Provavelmente os titulares mais importantes.

Cluster 1 — "Explosivo" (3 atletas)
Top Speed alta (27.03), Sprint Distance boa (62m), Distance razoável (7251m). Correm rápido mas em distâncias menores. Perfil de atacantes ou wing backs.

Cluster 2 — "Resistente" (12 atletas)
O grupo mais numeroso. Métricas equilibradas — boa distância (6451m), velocidade moderada. São os atletas que sustentam o jogo por mais tempo sem picos extremos.

Cluster 0 — "Baixa Intensidade" (9 atletas)
Menor em quase tudo — Distance 5493m, Top Speed 23.49, Session Load 703. Podem ser reservas, atletas com poucos minutos, ou em fase de recuperação.